# 2. Integration and Validation

Joins the tidy files from notebook 1 into the tables that go into PostgreSQL.

In [ ]:
import sys
import pandas as pd
import os

sys.path.append("..")
from src.utils import as_int, crime_type_table, add_other

CLEAN = "../data/clean"
OUT = "../data/db"

os.makedirs(OUT, exist_ok=True)

## 2.1 Master tables

The crime types tables are reference data written by hand, not
data extracted from the sources.

`crime_type_table` drops the `source_name` column, which exists only so that
`clean_crime_type` can recognise the Spanish spellings the Ministry publishes,
and with it the extra rows for years that worded a category differently.


In [ ]:
crime_type_mun = crime_type_table("mun")
crime_type_reg = crime_type_table("reg")

crime_type_mun.to_csv(f"{OUT}/crime_type_mun.csv", index=False)
crime_type_reg.to_csv(f"{OUT}/crime_type_reg.csv", index=False)

print("crime_type_mun")
display(crime_type_mun.head())
print("crime_type_reg")
display(crime_type_reg.head())

## 2.2 municipality, mun_population and mun_income

The `municipality` table contains one row per municipality with its INE
code, official name and surface area. The cleaned table is loaded, keeping the
ine_code as a string.

The `mun_population` table is also loaded from the cleaned dataset.


In [ ]:
municipality = pd.read_csv(f"{CLEAN}/municipalities.csv", dtype={"ine_code": str})
municipality.to_csv(f"{OUT}/municipality.csv", index=False)

mun_population = pd.read_csv(f"{CLEAN}/population.csv", dtype={"ine_code": str})
mun_population.to_csv(f"{OUT}/mun_population.csv", index=False)

print("municipality")
display(municipality.head())
print("mun_population")
display(mun_population.head())

## 2.3 mun_crime and reg_crime
The `crimes` file gives both the municipalities and a row for the whole region,
so it produces two tables.

A lookup dictionary is created
from the municipality table to map each municipality name to its corresponding
INE code. An assertion verifies that every municipality has been matched
successfully.

In [ ]:
crimes = pd.read_csv(f"{CLEAN}/crimes.csv")
reg_crime = crimes[crimes.is_region][["year", "crime_code", "crime_count"]]

mun_crime = crimes[~crimes.is_region].copy()
name_to_code = dict(zip(municipality["municipality_name"], municipality["ine_code"]))
mun_crime["ine_code"] = mun_crime.municipality_name.map(name_to_code)
assert mun_crime.ine_code.notna().all(), mun_crime[mun_crime.ine_code.isna()].match_name.unique()
mun_crime = mun_crime[["ine_code", "year", "crime_code", "crime_count"]]

print("reg_crime")
display(reg_crime.head())
print("mun_crime")
display(mun_crime.head())


### The 2020 problem

The 2020 file uses a shorter list of crime types. It has no split between
conventional crime and cybercrime, and instead of `OTHER_CONVENTIONAL`,
`CYBER_FRAUD` and `CYBER_OTHER` it publishes one column,
`Resto de infracciones penales`, which the master maps to `OTHER`.

That column is not a gap. It is exactly those three added together. So instead
of leaving 2020 with holes, or throwing the year away, or copying values from
other years, the same `OTHER` row is created for every other year by adding
up its parts. Every year then has the same eleven comparable types.

In [ ]:
mun_crime = add_other(mun_crime, ["ine_code", "year"])
reg_crime = add_other(reg_crime, ["year"])

as_int(mun_crime, ["crime_count"]).to_csv(f"{OUT}/mun_crime.csv", index=False)
as_int(reg_crime, ["crime_count"]).to_csv(f"{OUT}/reg_crime.csv", index=False)

print("reg_crime")
display(reg_crime.head())
print("mun_crime")
display(mun_crime.head())

## 2.4 reg_offences

`Recorded_offences` and `cleared_offences` are joined, so they
become two columns rather than two tables.

In [ ]:
recorded = pd.read_csv(f"{CLEAN}/recorded_offences.csv")
cleared = pd.read_csv(f"{CLEAN}/cleared_offences.csv")

reg_offences = (
    recorded.rename(columns={"crime_count": "recorded"}).merge(
    cleared.rename(columns={"crime_count": "cleared"}),
      on=["year", "crime_code"],
      how="outer"))


as_int(reg_offences, ["recorded", "cleared"]).to_csv(f"{OUT}/reg_offences.csv", index=False)
reg_offences.head()

## 2.5 reg_victims and reg_offenders

Kept as two tables, not one, because they do not have the same age groups.
`Victims` start at 0 and have an unknown age group, `offenders` start at 14

In [ ]:
reg_victims = pd.read_csv(f"{CLEAN}/victims.csv", dtype={"ine_code": str})
reg_offenders = pd.read_csv(f"{CLEAN}/offenders.csv", dtype={"ine_code": str})

as_int(reg_offenders, ["count"]).to_csv(f"{OUT}/reg_offenders.csv", index=False)
as_int(reg_victims, ["count"]).to_csv(f"{OUT}/reg_victims.csv", index=False)


print("reg_victims")
display(reg_victims.head())

print("reg_offenders")
display(reg_offenders.head())

### 9 tables in `data/db/`